In [2]:
from pymongo import MongoClient
import os
from dotenv import load_dotenv

load_dotenv()

client = MongoClient(os.getenv('MONGODB_URI'))

# Lister toutes les bases
db_list = client.list_database_names()

for db_name in db_list:
    print(f"\n📌 Base : {db_name}")
    db = client[db_name]

    # Lister les collections de cette base
    collections = db.list_collection_names()

    if not collections:
        print("  (aucune collection)")
        continue

    for col in collections:
        count = db[col].count_documents({})
        print(f"  - {col} : {count} documents")

db = client['flight_delay_history_db']

#cursor = db['aviationstack_historical_landed_flights'].find().limit(100)
#df_aviation = pd.DataFrame(list(cursor))
#df_aviation.head()
client.close()



📌 Base : flight_delay_db
  - aviationstack_flights : 1543 documents
  - afklm_flights : 1000 documents
  - weather_data : 78 documents

📌 Base : flight_delay_history_db
  - aviationstack_historical_landed_flights : 27 documents

📌 Base : flight_delays_test
  - test_collection : 22 documents

📌 Base : admin
  (aucune collection)

📌 Base : local
  - oplog.rs : 286 documents


In [7]:
#  exporter les données météo
import json
from pymongo import MongoClient
from dotenv import load_dotenv
import os

# Charger variables d'environnement
load_dotenv()

mongo_uri = os.getenv("MONGODB_URI")
client = MongoClient(mongo_uri)

db = client["flight_delay_db"]
collection = db["weather_data"]

# Récupérer tous les documents
docs = list(collection.find({}))

# Convertir ObjectId et datetime en string
def convert(o):
    if hasattr(o, "isoformat"):
        return o.isoformat()
    return str(o)

# Sauvegarde dans un fichier JSON
with open("weather_export.json", "w", encoding="utf-8") as f:
    json.dump(docs, f, default=convert, indent=4, ensure_ascii=False)

print(f"Export terminé : {len(docs)} documents sauvegardés dans weather_export.json")



Export terminé : 78 documents sauvegardés dans weather_export.json


In [6]:
# Petit script pour interroger la base flight_delay_history_db collection weather datd
from pymongo import MongoClient
from datetime import datetime
import os
from dotenv import load_dotenv

# Charger variables d'environnement
load_dotenv()

mongo_uri = os.getenv("MONGODB_URI")
client = MongoClient(mongo_uri)

db = client["flight_delay_db"]
collection = db["weather_data"]

doc = collection.find_one()
print(doc)


client.close()


{'_id': 'Paris_2025122222', 'base': 'stations', 'clouds': {'all': 0}, 'cod': 200, 'collected_at': datetime.datetime(2025, 12, 22, 22, 54, 7, 864000), 'coord': {'lon': 2.3488, 'lat': 48.8534}, 'dt': 1766443756, 'id': 2988507, 'main': {'temp': 8.55, 'feels_like': 6.42, 'temp_min': 7.25, 'temp_max': 9.01, 'pressure': 1010, 'humidity': 87, 'sea_level': 1010, 'grnd_level': 1000}, 'name': 'Paris', 'sys': {'type': 2, 'id': 2012208, 'country': 'FR', 'sunrise': 1766389313, 'sunset': 1766418989}, 'timezone': 3600, 'visibility': 10000, 'weather': [{'id': 800, 'main': 'Clear', 'description': 'clear sky', 'icon': '01n'}], 'wind': {'speed': 3.6, 'deg': 50}}


In [33]:
# test ML pas à pas
import os
from dotenv import load_dotenv
from pymongo import MongoClient
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
%matplotlib inline

# ============================================
# ETAPE 0  : RECUPERATION DES DONNEES
# ============================================
load_dotenv()

client = MongoClient(os.getenv("MONGODB_URI"))
db = client["flight_delay_history_db"]
collection = db["aviationstack_historical_landed_flights"]

# Filtre : vols atterris avec données complètes minimales
query = {
    "flight_status": "landed",
    "arrival.actual": {"$nin": [None, ""]},
    "arrival.scheduled": {"$nin": [None, ""]},
    "departure.scheduled": {"$nin": [None, ""]},
    "departure.actual": {"$nin": [None, ""]},
}

docs = list(collection.find(query))


# ============================================
# ETAPE 1  : APPLATISSEMENT DES DONNEES
# ============================================
# 🔹 Aplatir les documents imbriqués
df_flat = pd.json_normalize(
    docs,
    sep="_"  # airline.name -> airline_name
)

# 🔹 Garder / renommer les colonnes utiles
cols_to_keep = [
    "_id",
    "flight_date",
    "flight_status",
    "collected_at",
    "filtered_at",
    "live",

    # Airline
    "airline_name",
    "airline_iata",
    "airline_icao",

    # Aircraft
    "aircraft_registration",
    "aircraft_iata",
    "aircraft_icao",
    "aircraft_icao24",

    # Departure
    "departure_airport",
    "departure_timezone",
    "departure_iata",
    "departure_icao",
    "departure_terminal",
    "departure_gate",
    "departure_scheduled",
    "departure_estimated",
    "departure_actual",
    "departure_estimated_runway",
    "departure_actual_runway",

    # Arrival
    "arrival_airport",
    "arrival_timezone",
    "arrival_iata",
    "arrival_icao",
    "arrival_terminal",
    "arrival_gate",
    "arrival_baggage",
    "arrival_scheduled",
    "arrival_estimated",
    "arrival_actual",
    "arrival_estimated_runway",
    "arrival_actual_runway",

    # Flight
    "flight_number",
    "flight_iata",
    "flight_icao",
    "flight_codeshared",
]

# Certaines colonnes peuvent ne pas exister pour tous les docs → on intersecte
existing_cols = [c for c in cols_to_keep if c in df_flat.columns]
df_flat = df_flat[existing_cols]

# Optionnel : conversion de quelques champs en datetime
datetime_cols = [
    "flight_date",
    "collected_at",
    "filtered_at",
    "departure_scheduled",
    "departure_estimated",
    "departure_actual",
    "departure_estimated_runway",
    "departure_actual_runway",
    "arrival_scheduled",
    "arrival_estimated",
    "arrival_actual",
    "arrival_estimated_runway",
    "arrival_actual_runway",
]

for col in datetime_cols:
    if col in df_flat.columns:
        df_flat[col] = pd.to_datetime(df_flat[col], errors="coerce")

# df_flat.head()

# ============================================
# ETAPE 2  : ANALYSE DES VALEURS NULLS ET SUPPRESSIONS DES COLONNES INUTILES
# ============================================
# Nombre de valeurs nulles par colonne
null_counts = df_flat.isnull().sum()

# Pourcentage de valeurs nulles par colonne
null_percent = (null_counts / len(df_flat)) * 100

# Résumé filtré : uniquement les colonnes avec plus de 30% de valeurs nulles
null_summary = (
    pd.DataFrame({
        "null_count": null_counts,
        "null_percent": null_percent.round(2)
    })
    .query("null_percent > 0")
    .sort_values(by="null_percent", ascending=False)
)

display(null_summary)

# On supprimme les colonnes qui ont plus de 30% de valeurs nulls
# Colonnes à supprimer : null_percent > 30%
cols_to_drop = null_summary[null_summary["null_percent"] > 30.0].index.tolist()

# Nouveau DataFrame nettoyé
df_flat_filtered = df_flat.drop(columns=cols_to_drop)

print(f"Colonnes supprimées : {len(cols_to_drop)}")
print(cols_to_drop)

# Suppressions des colonnes runway car inutiles
cols_runway = [col for col in df_flat_filtered.columns if "runway" in col.lower()]

df_flat_filtered = df_flat_filtered.drop(columns=cols_runway)

print("Colonnes runway supprimées :", cols_runway)

# df_flat_filtered.head()

# ============================================
# ETAPE 3  : TRAITEMENT DES VALEURS NULLS
# ============================================
# Ici c'est la colonne departure_terminal
display(df_flat_filtered['departure_terminal'].value_counts()) 
mode_value = df_flat_filtered['departure_terminal'].mode()[0]
df_flat_filtered['departure_terminal'] = df_flat_filtered['departure_terminal'].fillna(mode_value)


# ============================================
# ETAPE 4 : CALCUL DES DÉLAIS
# ============================================
# Fonction utilitaire pour calculer un délai en minutes
def compute_delay(actual, scheduled):
    if pd.isna(actual) or pd.isna(scheduled):
        return None
    delay = (actual - scheduled).total_seconds() / 60
    return max(delay, 0)  # clamp à 0

# Délais départ
df_flat_filtered["departure_delay_actual"] = df_flat_filtered.apply(
    lambda row: compute_delay(row["departure_actual"], row["departure_scheduled"]),
    axis=1
)

df_flat_filtered["departure_delay_estimated"] = df_flat_filtered.apply(
    lambda row: compute_delay(row["departure_estimated"], row["departure_scheduled"]),
    axis=1
)

# Délais arrivée
df_flat_filtered["arrival_delay_actual"] = df_flat_filtered.apply(
    lambda row: compute_delay(row["arrival_actual"], row["arrival_scheduled"]),
    axis=1
)

df_flat_filtered["arrival_delay_estimated"] = df_flat_filtered.apply(
    lambda row: compute_delay(row["arrival_estimated"], row["arrival_scheduled"]),
    axis=1
)

def compute_duration(dep, arr):
    if pd.isna(dep) or pd.isna(arr):
        return None
    duration = (arr - dep).total_seconds() / 60
    return max(duration, 0)  # clamp à 0 pour éviter les valeurs négatives

df_flat_filtered["flight_duration_scheduled"] = df_flat_filtered.apply(
    lambda row: compute_duration(row["departure_scheduled"], row["arrival_scheduled"]),
    axis=1
)


# Vérification rapide
df_flat_filtered[[
    "departure_delay_actual",
    "departure_delay_estimated",
    "arrival_delay_actual",
    "arrival_delay_estimated",
    "flight_duration_scheduled"
]].sort_values(by="arrival_delay_actual", ascending=False).head()

#df_flat_filtered.sort_values(by="arrival_delay_actual", ascending=False).head()


# ============================================
# ETAPE 5 : DONNEES METEO
# ============================================
# Normalisation des dates vols en UTC (une seule fois, propre)
for col in ["departure_actual", "departure_estimated", "departure_scheduled"]:
    if col in df_flat_filtered.columns:
        df_flat_filtered[col] = pd.to_datetime(df_flat_filtered[col], errors="coerce", utc=True)

# --- 1. Mapping simple IATA -> Ville météo ---
airport_to_city = {
    "CDG": "Paris",
    "ORY": "Paris",
    "AMS": "Amsterdam",
    "LHR": "London",
    "JFK": "New York"
}

# --- 2. Connexion à la base météo ---
db_weather = client["flight_delay_db"]
weather_col = db_weather["weather_data"]


# --- 3. Fonction pour trouver la météo la plus proche dans ± 6h ---
def find_closest_weather(city, target_dt):
    if pd.isna(target_dt):
        return None, None

    # target_dt est déjà en UTC (utc=True plus haut)
    # Fenêtre ± 6h
    start = target_dt - pd.Timedelta(hours=6)
    end = target_dt + pd.Timedelta(hours=6)

    docs = list(weather_col.find({
        "name": city,
        "collected_at": {
            "$gte": start.to_pydatetime(),
            "$lte": end.to_pydatetime()
        }
    }).sort("collected_at", 1))

    if not docs:
        return None, None

    def time_diff(doc):
        collected = pd.to_datetime(doc["collected_at"], utc=True)
        return abs(collected - target_dt)

    closest = min(docs, key=time_diff)

    description = closest["weather"][0]["description"]
    date_used = closest["collected_at"]

    return description, date_used

# --- 4. Colonnes à ajouter ---
df_flat_filtered["departure_meteo"] = None
df_flat_filtered["departure_meteo_date"] = None

# --- 5. Application ligne par ligne ---
for idx, row in df_flat_filtered.iterrows():

    # 1. Trouver la ville météo
    iata = row.get("departure_iata")
    city = airport_to_city.get(iata)

    if city is None:
        continue

    # 2. Choisir la meilleure date de référence
    if pd.notna(row["departure_actual"]):
        ref_dt = row["departure_actual"]
    elif pd.notna(row["departure_estimated"]):
        ref_dt = row["departure_estimated"]
    else:
        ref_dt = row["departure_scheduled"]

    # 3. Chercher la météo
    meteo, meteo_dt = find_closest_weather(city, ref_dt)

    df_flat_filtered.at[idx, "departure_meteo"] = meteo
    df_flat_filtered.at[idx, "departure_meteo_date"] = meteo_dt

print("Ajout des colonnes météo terminé.")

# traitement des météos nulls
missing_count = df_flat_filtered["departure_meteo"].isna().sum()
total = len(df_flat_filtered)
missing_percent = (missing_count / total) * 100

print(f"Valeurs manquantes : {missing_count} / {total}")
print(f"Pourcentage manquant : {missing_percent:.2f}%")
print(df_flat_filtered["departure_meteo"].value_counts(dropna=False))

mode_meteo = df_flat_filtered["departure_meteo"].mode()[0]
df_flat_filtered["departure_meteo"] = df_flat_filtered["departure_meteo"].fillna(mode_meteo)

print("Mode utilisé pour remplir les valeurs manquantes :", mode_meteo)
#df_flat_filtered.sort_values(by="arrival_delay_actual", ascending=False).head()

# Vérification si il y a encore des valeurs nulls
df_flat_filtered.isnull().any().any()

# ============================================
# ETAPE 6 : SUPPRESSION DES COLONNES INUTILES ET TRAITEMENT DES DERNIERS VALEURS NULLS
# ============================================
# 1. Colonnes contenant "date"
cols_with_date = [c for c in df_flat_filtered.columns if "date" in c.lower()]

# 2. Colonnes explicitement inutiles
explicit_drop = [
    "flight_status",
    "collected_at",
    "filtered_at",
    "departure_scheduled",
    "departure_estimated",
    "departure_actual",
    "arrival_scheduled",
    "arrival_estimated",
    "arrival_actual",
    "departure_timezone"
]

# 3. Autres colonnes non pertinentes pour le ML
other_useless = [
    "airline_icao",
    "aircraft_icao",
    "aircraft_icao24",
    "flight_icao",
    "flight_number",
    "flight_iata",
    "flight_codeshared",
    "arrival_terminal",
    "arrival_gate",
    "arrival_timezone",
    "arrival_baggage",
    "departure_terminal",
    "departure_gate",
    "departure_airport",
    "arrival_airport",
    "departure_icao",
    "arrival_icao",
    "airline_iata",
]

# Fusionner toutes les colonnes à supprimer
cols_to_remove = set(cols_with_date + explicit_drop + other_useless)

# Garder uniquement celles qui existent réellement
cols_to_remove = [c for c in cols_to_remove if c in df_flat_filtered.columns]

# Suppression
df_flat_filtered = df_flat_filtered.drop(columns=cols_to_remove)

print("Colonnes supprimées :", cols_to_remove)
print("Nombre total supprimé :", len(cols_to_remove))

# Vérification si il y a encore des valeurs nulls
# df_flat_filtered.isnull().any().any()
df_flat_filtered.sort_values(by="arrival_delay_actual", ascending=False).head()



,null_count,null_percent
live,47,100.00
aircraft_registration,47,100.00
aircraft_iata,47,100.00
aircraft_icao,47,100.00
flight_codeshared,47,100.00
filtered_at,45,95.74
arrival_gate,45,95.74
departure_gate,30,63.83
arrival_baggage,23,48.94
departure_terminal,11,23.40


Colonnes supprimées : 9
['live', 'aircraft_registration', 'aircraft_iata', 'aircraft_icao', 'flight_codeshared', 'filtered_at', 'arrival_gate', 'departure_gate', 'arrival_baggage']
Colonnes runway supprimées : ['departure_estimated_runway', 'departure_actual_runway', 'arrival_estimated_runway', 'arrival_actual_runway']


departure_terminal
1     20
3      5
2E     5
2F     4
4      1
5      1
Name: count, dtype: int64

Ajout des colonnes météo terminé.
Valeurs manquantes : 23 / 47
Pourcentage manquant : 48.94%
departure_meteo
None             23
clear sky        23
broken clouds     1
Name: count, dtype: int64
Mode utilisé pour remplir les valeurs manquantes : clear sky
Colonnes supprimées : ['airline_icao', 'flight_date', 'flight_icao', 'flight_iata', 'departure_terminal', 'departure_actual', 'aircraft_icao24', 'departure_estimated', 'arrival_terminal', 'departure_meteo_date', 'arrival_icao', 'departure_icao', 'arrival_estimated', 'airline_iata', 'collected_at', 'arrival_timezone', 'arrival_scheduled', 'flight_status', 'arrival_airport', 'arrival_actual', 'departure_timezone', 'flight_number', 'departure_scheduled', 'departure_airport']
Nombre total supprimé : 24


,_id,airline_name,departure_iata,arrival_iata,departure_delay_actual,departure_delay_estimated,arrival_delay_actual,arrival_delay_estimated,flight_duration_scheduled,departure_meteo
23,U24184_2025-12-22,easyJet,ORY,NAP,39.0,0.0,21.0,21.0,130.0,clear sky
26,U29043_2026-02-03,easyJet,ORY,LIS,54.0,0.0,20.0,18.0,95.0,broken clouds
36,AF_2026-02-04,Air France,ORY,CDG,14.0,0.0,19.0,21.0,15.0,clear sky
19,XK787_2025-12-22,Air Corsica,ORY,BIA,9.0,0.0,9.0,3.0,95.0,clear sky
12,AF4466_2025-12-22,Air France,ORY,BIA,9.0,0.0,9.0,3.0,95.0,clear sky
